# 00 — Data Pipeline

**Before running:** point `METRO_DATA_DIR` at the `Metro_Data` folder (it
should contain `HZMETRO/`, `SHMETRO/`, `BEIJING_SUBWAY_15MIN/` subfolders with
`.dyna`/`.rel`/`.geo` files).

In [ ]:
import sys, os

candidates = [os.path.abspath("."), os.path.abspath("..")]
found = next((c for c in candidates
              if os.path.isfile(os.path.join(c, "cahgtn", "__init__.py"))), None)
if found is None:
    raise ModuleNotFoundError(
        f"Could not find a 'cahgtn' package folder in {candidates[0]} or "
        f"{candidates[1]}. See the folder layout in the markdown cell above."
    )
sys.path.insert(0, found)
print(f"Using cahgtn package from: {os.path.join(found, 'cahgtn')}")

import numpy as np
from cahgtn.data_pipeline import load_city, build_seasonal_profile, build_windows

METRO_DATA_DIR = "Metro_Data"      # <-- edit to your local path
CITY = "Beijing"                  # "Hangzhou" | "Shanghai" | "Beijing"

data = load_city(METRO_DATA_DIR, CITY)
print(f"{CITY}: {data.num_stations} stations, {data.flow.shape[0]} timesteps, "
      f"{data.n_slots} slots/day (daily_lag={data.daily_lag})")
for k, v in data.idx.items():
    print(f"  {k:>14}: {len(v)} timesteps")

In [ ]:
profile_internal = build_seasonal_profile(data, data.idx["internal_fit"])
profile_official  = build_seasonal_profile(data, data.idx["train"])

X_ifit,  Y_ifit,  TF_ifit,  EX_ifit,  SEAS_ifit  = build_windows(data, data.idx["internal_fit"], profile_internal)
X_ival,  Y_ival,  TF_ival,  EX_ival,  SEAS_ival  = build_windows(data, data.idx["internal_val"], profile_internal)
X_train, Y_train, TF_train, EX_train, SEAS_train = build_windows(data, data.idx["train"],        profile_official)
X_test,  Y_test,  TF_test,  EX_test,  SEAS_test  = build_windows(data, data.idx["test"],         profile_official)

print("internal_fit ", X_ifit.shape,  Y_ifit.shape, SEAS_ifit.shape)
print("internal_val ", X_ival.shape,  Y_ival.shape, SEAS_ival.shape)
print("official train", X_train.shape, Y_train.shape, SEAS_train.shape)
print("official test ", X_test.shape,  Y_test.shape, SEAS_test.shape)

In [ ]:
naive_test = SEAS_test
print("naive_test", naive_test.shape)

## Save Results

In [ ]:
sim_precomputed = (data.adjacency_similarity_precomputed.numpy() if data.adjacency_similarity_precomputed is not None
                    else np.zeros((data.num_stations, data.num_stations), dtype=np.float32))
corr_precomputed = (data.adjacency_correlation_precomputed.numpy() if data.adjacency_correlation_precomputed is not None
                     else np.zeros((data.num_stations, data.num_stations), dtype=np.float32))

np.savez_compressed(
    f"{CITY}_windows.npz",
    X_ifit=X_ifit, Y_ifit=Y_ifit, TF_ifit=TF_ifit, EX_ifit=EX_ifit, SEAS_ifit=SEAS_ifit,
    X_ival=X_ival, Y_ival=Y_ival, TF_ival=TF_ival, EX_ival=EX_ival, SEAS_ival=SEAS_ival,
    X_train=X_train, Y_train=Y_train, TF_train=TF_train, EX_train=EX_train, SEAS_train=SEAS_train,
    X_test=X_test, Y_test=Y_test, TF_test=TF_test, EX_test=EX_test, SEAS_test=SEAS_test,
    naive_test=naive_test,
    adjacency_physical=data.adjacency_physical.numpy(),
    adjacency_corr=data.adjacency_corr.numpy(),
    adjacency_similarity_precomputed=sim_precomputed,
    adjacency_correlation_precomputed=corr_precomputed,
    num_stations=data.num_stations,
)
print(f"Saved {CITY}_windows.npz -- proceed to 01_Train_CAHGTN.ipynb for this city,")
print(f"then come back here with CITY set to the next one.")